# Lactanet Genetics - Complete Guided Analysis

This is the **single, clean version** of the project. Run every cell in order,
top to bottom, with **Kernel → Restart & Run All**. Do not add other loading
cells above or between these - duplicated loading cells were the cause of
earlier errors (missing provinces, `KeyError`, etc.).

**Folder needed:** place your 10 `Lactanet Genetics (XX).xlsx` files (or the
`Lactanet_Genetics_XX.xlsx` style - both work) plus `diccionario_lactanet.xlsx`
inside `data/raw/`, next to this notebook.


## Step 0 - Setup

Import libraries and set up the folder path.


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

DATA_DIR = Path("data/raw")
pd.set_option("display.max_columns", 40)
sns.set_style("whitegrid")

CURRENT_YEAR = 2026


## Step 1 - Load all provincial files

Recognizes **both** file-naming styles:
- `Lactanet Genetics (AB).xlsx`
- `Lactanet_Genetics_AB.xlsx` / `Lactanet_Genetics_AB__Copie.xlsx`

If a province is missing from the printed list below, the file is not in
`data/raw/` - check the folder before continuing.


In [ ]:
PROVINCE_NAMES = {
    "AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba",
    "NB": "New Brunswick", "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia", "ON": "Ontario", "PEI": "Prince Edward Island",
    "QC": "Quebec", "SK": "Saskatchewan",
}

PATTERN_PARENS = re.compile(r"\(([A-Za-z]+)\)")
PATTERN_UNDERSCORE = re.compile(r"Lactanet[_ ]Genetics[_ ]([A-Za-z]+)", re.IGNORECASE)

# Some provinces use a longer alias with no separator, e.g. "GeneticsABT.xlsx",
# "GeneticsONT.xlsx" -- map those aliases to the standard 2-3 letter codes.
CODE_ALIASES = {"ABT": "AB", "ONT": "ON", "PI": "PEI", "QUE": "QC", "SAS": "SK", "MAN": "MB"}

def extract_province_code(file_path: Path) -> str:
    stem = file_path.stem

    match = PATTERN_PARENS.search(stem) or PATTERN_UNDERSCORE.search(stem)
    if match:
        code_ = match.group(1).upper()
    else:
        # Fallback: strip out the known filler words and whatever remains is the code
        candidate = stem
        for word in ["Lactanet", "Genetics", "Copie"]:
            candidate = re.sub(word, "", candidate, flags=re.IGNORECASE)
        code_ = candidate.strip("_ ()").upper()

    code_ = CODE_ALIASES.get(code_, code_)

    if not code_ or code_ not in PROVINCE_NAMES:
        raise ValueError(f"Unrecognized province code '{code_}' in file: {file_path.name}")
    return code_

files = sorted(
    f for f in DATA_DIR.glob("*.xlsx")
    if "lactanet" in f.name.lower()
    and "genetics" in f.name.lower()
    and not f.name.startswith("~$")
)

print(f"Found {len(files)} provincial files (expected: 10):")
for f in files:
    print(" -", f.name, "->", extract_province_code(f))


In [ ]:
frames = []
for file_path in files:
    province_code = extract_province_code(file_path)
    df_province = pd.read_excel(file_path, sheet_name=0, header=1, engine="openpyxl")
    df_province.insert(0, "Province", province_code)
    df_province.insert(1, "Province Name", PROVINCE_NAMES[province_code])
    frames.append(df_province)
    print(f"{file_path.name}: {len(df_province)} records loaded ({province_code})")

print(f"\nTotal rows loaded: {sum(len(f) for f in frames)}")


## Step 2 - Sort and consolidate

Combine the 10 tables into one, sorted by **LPI** (Lifetime Performance
Index, the main overall ranking index) from highest to lowest.


In [ ]:
raw = pd.concat(frames, ignore_index=True)
consolidated = raw.sort_values("LPI", ascending=False).reset_index(drop=True)

print(f"Shape: {consolidated.shape}")
print(f"Provinces present: {sorted(consolidated['Province Name'].unique())}")
consolidated.head(10)


## Step 3 - Understand the variables

Load the dictionary that explains every column.


In [ ]:
dictionary_path = DATA_DIR / "diccionario_lactanet.xlsx"
dictionary = pd.read_excel(dictionary_path, engine="openpyxl")
dictionary


## Step 4 - Clean the data

- Coerce numeric columns to actual numbers.
- Turn `Act.` / `GS` letter-codes into clear `Is Active` / `Is Genomic` flags.
- Strip whitespace from text columns.


In [ ]:
clean = consolidated.copy()

text_cols = clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    clean[c] = clean[c].astype("string").str.strip()

NUMERIC_COLS = [
    "LPI", "PI", "LTI", "HWI", "RI", "MI", "EI", "Pro$", "Milk", "Fat", "Prot",
    "%F", "%P", "ME", "FE", "BMR", "MR", "SCS", "Conf", "MS", "F&L", "DS", "RP",
    "%R", "REL_PROT", "REL_CONF",
]
clean[NUMERIC_COLS] = clean[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")

clean["Is Active"] = clean["Act."].fillna("").eq("A").astype(bool)
clean["Is Genomic"] = clean["GS"].fillna("").eq("G").astype(bool)
clean["LPI Code"] = clean["LPI Code"].astype(str).str.strip()

print(f"Active animals: {clean['Is Active'].sum()} / {len(clean)}")
print(f"Genomically tested animals: {clean['Is Genomic'].sum()} / {len(clean)}")
clean.head()


### Data notes to keep in mind

- **These are the top 400 animals per province by LPI**, not a full census -
  provincial averages describe each province's *best-ranked* animals, not
  its whole population. Every average in this notebook is a **dataset
  average** (computed from these 4,000 records only) - it is not an
  official national average across all Canadian dairy cattle.
- **Every one of the 4,000 animals in this dataset is unique** - no animal
  appears in more than one province's file. This was checked directly
  (pairwise ID comparison across all 10 provinces, 0 overlaps found), not
  assumed.
- **This is a snapshot in time, not a static or permanent dataset.**
  Lactanet's official genetic evaluations are released periodically
  (multiple times per year), and which animals rank in each province's
  top 400 changes as new births, milk recording, classification, and
  genomic results are added. The selection criterion for inclusion in
  each provincial file is **LPI at the time of the data pull** - the same
  animal could rank in or out of a future top-400 list as new evaluations
  are released. Every figure in this notebook describes this snapshot
  specifically, not a fixed or permanent ranking.

**Official definitions:**

- **`LPI Code`**: `EBV` means the animal has an *official* published index
  for **both** production and conformation, based on its own or its
  daughters' phenotype/classification records - this can be a traditional
  EBV or a genomic EBV (GEBV). `PA` (Pedigree Average / Genomic PA) means
  the animal lacks an official index in at least one of those two areas -
  this is not limited to young heifers; a mature cow can also be `PA` if
  she's missing official production or classification data for other
  reasons.
- **`GS = G` (genotyped) does not automatically mean `LPI Code = EBV`.**
  An animal can have a genomic test on file and still be `PA` if it hasn't
  yet met the phenotype/production requirements - this is expected and
  normal, not a data quality issue.
- **`%R` is NOT a reliability measure.** It stands for **Relationship
  Percent** - the animal's genetic relationship to the breed population
  (Canada's equivalent of Expected Future Inbreeding, typical range
  15-23%), used to monitor and prevent inbreeding. `REL_PROT` and
  `REL_CONF` (separate columns) are the actual reliability measures used
  elsewhere in this analysis.
- **`LPI Code` now shows a plausible PA/EBV split in all 10 provinces**
  (checked directly) - no province needs to be excluded from PA-vs-EBV
  comparisons.
- **`Act.` = inactive** mostly matches animals born in the current year
  (2026) - likely newly registered calves not yet active in milk recording,
  not deceased animals. This is a pattern in the data, not an official
  confirmation from Lactanet.
- **The "2+ years old" filter used below** is an approximation for "should
  have had the chance to reach EBV status by now" - but per the official
  definition above, a mature animal can still legitimately be `PA` for
  reasons unrelated to age, so this filter is a reasonable proxy, not a
  guarantee.
- **`GS` (genomic tested) reflects whether a result is currently on file,
  not necessarily whether an animal was ever tested** - a recently sampled
  animal awaiting results (which can take about a month) looks identical to
  one never tested. This mainly affects the youngest birth-year cohorts.


## Step 4.1 - What information does each animal actually have?

Before comparing provinces on any trait, it matters what *kind* of evidence
is behind each animal's number. Two fields determine this:

- **`LPI Code`**: `EBV` = official index built from real production/type
  records (own or daughters'). `PA` = pedigree average only, no official
  index in at least one area.
- **`GS`**: `G` = has a DNA/genomic test on file.

Crossing them gives four genuinely different information states, not just
an age or data-quality distinction.


In [ ]:
def information_group(row):
    if row["LPI Code"] == "PA" and not row["Is Genomic"]:
        return "1. PA only (pedigree average, no DNA)"
    elif row["LPI Code"] == "PA" and row["Is Genomic"]:
        return "2. G+PA (genomic potential, no phenotype yet)"
    elif row["LPI Code"] == "EBV" and not row["Is Genomic"]:
        return "3. EBV, no G (confirmed production/type, no DNA)"
    else:
        return "4. EBV+G (confirmed production/type AND genomic)"

clean["Information Group"] = clean.apply(information_group, axis=1)

group_summary = clean.groupby("Information Group", observed=True).agg(
    Count=("LPI", "size"),
    Avg_REL_PROT=("REL_PROT", "mean"),
    Avg_REL_CONF=("REL_CONF", "mean"),
).round(1)
group_summary


**Reading this table:** reliability climbs from PA-only (weakest
evidence) to EBV+G (strongest) - confirming the four groups really do carry
different amounts of information, using the dataset's own reliability
columns rather than an assumption. Notably, **G+PA animals average higher
`REL` than EBV-without-G animals** - meaning genomic predictions currently
have higher reliability than phenotype-only animals in this dataset. This
is a statement about precision, not about which information source
matters more in a biological or causal sense.


In [ ]:
information_by_province = clean.groupby(["Province Name", "Information Group"], observed=True).size().unstack(fill_value=0)
information_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
information_by_province.plot(kind="barh", stacked=True, ax=ax, color=["#B23A3A", "#C9A66B", "#8FA9A0", "#2E5B4D"])
ax.set_xlabel("Number of animals (out of 400)")
ax.set_title("Information composition by province: how much evidence backs each animal's number?")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


**Findings:** all 10 provinces show a believable mix across all four
groups, with the EBV+G group (the most complete information) ranging from
just 2 animals (Manitoba) to 67 (Ontario), a real difference in how much
can be confidently said about each province using proven-only data.


## Step 4.1.1 - Robustness check: is this gap just an age effect?

The table above mixes animals of very different ages: `EBV` and `EBV+G`
animals are always 2+ years old (confirmed directly, 0 animals with an
official phenotype-based evaluation are under 2 years), but `PA only` and
`G+PA` include large numbers of very young animals simply because they
haven't had time to accumulate phenotype records yet, not necessarily
because they were passed over. Restricting all four groups to animals 2+
years old checks whether the reliability gap survives an age-controlled
comparison.


In [ ]:
CURRENT_YEAR = 2026
clean["Age (years, approx.)"] = CURRENT_YEAR - clean["Birth Year"]
mature_check = clean[clean["Age (years, approx.)"] >= 2]

age_controlled = mature_check.groupby("Information Group", observed=True).agg(
    N=("REL_PROT", "size"),
    Avg_REL_PROT=("REL_PROT", "mean"),
    Avg_REL_CONF=("REL_CONF", "mean"),
).round(1)

print("Reliability by group, animals 2+ years only (age-controlled):")
age_controlled


**Findings:** the gap survives the age control. Even comparing only
same-age (2+ year old) animals, `PA only` remains far less reliable
(around 42%) than `EBV+G` (around 84%), so the earlier finding is not
simply an artifact of younger animals dragging down the PA-only average.
The sample size for mature PA-only animals is smaller than the all-ages
figure (roughly 90 vs. over 1,000), a genuine reduction worth keeping in
mind, but still large enough to support the comparison.


## Step 4.2 - Sensitivity analysis: does animal-level reliability change the picture?

**Primary result throughout this notebook: the unweighted mean of each
province's 400 animals.** The top-400 list is a real, enumerated group,
not a sample used to infer a broader population - the simple mean directly
answers "what is the average official value across the animals on this
list," which is the question provincial comparisons in this notebook ask.
This also matches standard practice in dairy genetics reporting (herd
averages and genetic trend graphs are conventionally unweighted means of
published EBVs/PTAs).

Since animals differ in how reliable their individual estimates are, this
step checks - as a **sensitivity analysis, not a replacement for the
primary result** - whether weighting each animal by its own
`REL_PROT`/`REL_CONF` would change the conclusion. This is not a standard
industry procedure for computing group means (Interbull/BLUP theory uses
reliability to combine information about the *same* animal, not to weight
*different* animals against each other in a group average) - it's an
exploratory check of robustness, and is reported as one.


In [ ]:
import numpy as np

def weighted_mean(group, value_col, weight_col):
    weights = group[weight_col].fillna(0)
    values = group[value_col]
    mask = values.notna() & (weights > 0)
    return np.average(values[mask], weights=weights[mask]) if mask.sum() > 0 else np.nan

def weighted_standard_error(group, value_col, weight_col):
    weights = group[weight_col].fillna(0)
    values = group[value_col]
    mask = values.notna() & (weights > 0)
    v, w = values[mask], weights[mask]
    wmean = np.average(v, weights=w)
    variance = np.average((v - wmean) ** 2, weights=w)
    effective_n = (w.sum() ** 2) / (w ** 2).sum() # accounts for unequal weighting
    return np.sqrt(variance / effective_n), effective_n

comparison_rows = []
for province, group in clean.groupby("Province Name", observed=True):
    ebv_g_only = group[(group["LPI Code"] == "EBV") & (group["Is Genomic"])]
    track_a_mean = ebv_g_only["Conf"].mean() if len(ebv_g_only) > 0 else np.nan
    unweighted_mean = group["Conf"].mean()
    weighted, eff_n = weighted_mean(group, "Conf", "REL_CONF"), weighted_standard_error(group, "Conf", "REL_CONF")[1]

    comparison_rows.append({
        "Province": province,
        "PRIMARY: Unweighted mean (Conf, all 400)": round(unweighted_mean, 2),
        "Sensitivity: Reliability-weighted mean": round(weighted, 2),
        "Difference (sensitivity - primary)": round(weighted - unweighted_mean, 2),
        "Reference only: Track A mean (EBV+G, n=%d)" % len(ebv_g_only): round(track_a_mean, 2) if not np.isnan(track_a_mean) else None,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["Sensitive to method? (diff > 1 point)"] = comparison_df["Difference (sensitivity - primary)"].abs() > 1.0
comparison_df.sort_values("PRIMARY: Unweighted mean (Conf, all 400)", ascending=False)


**How to read this:** the unweighted mean (primary result) and the
reliability-weighted mean (sensitivity check) agree closely for most
provinces - that agreement is itself evidence the unweighted result is
robust. **Manitoba and Prince Edward Island show the largest differences**
between the two approaches, so any conclusion about their conformation
levels specifically should be treated with more caution than for provinces
where both approaches agree. The Track A (EBV+G-only) column is shown only
as a third reference point - with n as low as 2 (Manitoba), it illustrates
how unstable a very small subgroup mean can be, which is exactly why it
isn't used as the primary or sensitivity estimator here.


## Step 5 - National and provincial averages


In [ ]:
KEY_INDICES = ["LPI", "PI", "Pro$", "Milk", "Fat", "Prot", "Conf", "%R"]

national_average = clean[KEY_INDICES].mean().round(2)
print("National average:")
print(national_average)


In [ ]:
by_province = clean.groupby("Province Name", observed=True)[KEY_INDICES].mean().round(2)
by_province["Animal Count"] = clean.groupby("Province Name", observed=True).size()
by_province = by_province.sort_values("LPI", ascending=False)
by_province


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
by_province["LPI"].plot(kind="bar", ax=ax, color="#2E5B4D")
ax.axhline(national_average["LPI"], color="red", linestyle="--", label="National Average")
ax.set_ylabel("Average LPI")
ax.set_xlabel("Province")
ax.set_title("Average LPI by province vs. national average")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Step 6 - Genomic testing (% genotyped by province)

`GS = G` means the animal has a genomic (DNA) test on file.


In [ ]:
genomic_summary = clean.groupby("Province Name", observed=True)["Is Genomic"].agg(
    Genotyped="sum", Total="count"
)
genomic_summary["Not Genotyped"] = genomic_summary["Total"] - genomic_summary["Genotyped"]
genomic_summary["% Genotyped"] = (genomic_summary["Genotyped"] / genomic_summary["Total"] * 100).round(1)
genomic_summary["% Not Genotyped"] = (100 - genomic_summary["% Genotyped"]).round(1)
genomic_summary = genomic_summary.sort_values("% Genotyped", ascending=False)
genomic_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
provinces = genomic_summary.index
genotyped = genomic_summary["% Genotyped"]
not_genotyped = genomic_summary["% Not Genotyped"]

ax.bar(provinces, genotyped, label="Genotyped (GS = G)", color="#2E5B4D")
ax.bar(provinces, not_genotyped, bottom=genotyped, label="Not genotyped", color="#C9A66B")
for i, pct in enumerate(genotyped):
    ax.text(i, pct / 2, f"{pct}%", ha="center", va="center", color="white", fontweight="bold")

ax.set_ylabel("% of animals")
ax.set_xlabel("Province")
ax.set_title("Genotyped vs. not genotyped animals, by province")
ax.set_ylim(0, 100)
ax.legend(loc="upper right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Step 7 - Filter animals 2+ years old

We only have birth *year*, not the exact date, so age is approximate (whole
calendar years). From here on, `mature` = animals that should already be old
enough to be in production.


In [ ]:
clean["Age (years, approx.)"] = CURRENT_YEAR - clean["Birth Year"]
mature = clean[clean["Age (years, approx.)"] >= 2].copy()
print(f"Animals 2+ years old: {len(mature)} of {len(clean)}")

ebv = mature[mature["LPI Code"] == "EBV"].copy()
print(f"Of those, with own EBV data: {len(ebv)}")


## Step 8 - Highest KG and % components by province

Best EBV animal per province for Milk/Fat/Protein (kg) and %Fat/%Protein.
Using EBV-only animals keeps the comparison fair (own data, not just
pedigree).


In [ ]:
ebv["KG Score"] = ebv["Fat"] + ebv["Prot"]
top_kg_by_province = (
    ebv.sort_values(["Province Name", "KG Score"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .first()[["Name", "Milk", "Fat", "Prot"]]
)

ebv["PCT Score"] = ebv["%F"] + ebv["%P"]
top_pct_by_province = (
    ebv.sort_values(["Province Name", "PCT Score"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .first()[["Name", "%F", "%P"]]
)

print("Top KG components by province:")
print(top_kg_by_province)
print("\nTop % components by province:")
print(top_pct_by_province)


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 9))

top_kg_by_province[["Milk", "Fat", "Prot"]].plot(kind="bar", ax=ax1, color=["#8FA9A0", "#C9A66B", "#2E5B4D"])
ax1.set_title("Best EBV animal per province - Milk, Fat, Protein (kg)")
ax1.set_ylabel("kg")
ax1.tick_params(axis="x", rotation=45)

top_pct_by_province[["%F", "%P"]].plot(kind="bar", ax=ax2, color=["#C9A66B", "#2E5B4D"])
ax2.set_title("Best EBV animal per province - %Fat, %Protein")
ax2.set_ylabel("%")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


**Findings:** Kg leaders and % leaders are often *different animals* - a
cow can lead in yield (kg) without leading in concentration (%). Some top
animals (e.g. in Alberta and Ontario) are identical, confirming the earlier
note that top-ranked animals are shared across provincial files.


## Step 9 - Best conformation animal by province


In [ ]:
top_conf_by_province = (
    ebv.sort_values(["Province Name", "Conf"], ascending=[True, False])
    .groupby("Province Name", observed=True)
    .first()[["Name", "Conf", "MS", "F&L", "DS", "RP"]]
)
top_conf_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_conf_by_province["Conf"].sort_values(ascending=False).plot(kind="bar", ax=ax, color="#2E5B4D")
ax.set_title("Best conformation score (Conf) by province - EBV animals")
ax.set_ylabel("Conformation score")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## Step 10 - Strengths and weaknesses by province (conformation)

Green bar = above the national average for that trait (strength).
Red bar = below the national average (weakness). No heatmap - the province
name sits right on the bar so it's readable at a glance.


In [ ]:
CONFORMATION = ["Conf", "MS", "F&L", "DS", "RP"]

conf_by_province = ebv.groupby("Province Name", observed=True)[CONFORMATION].mean()
national_avg_conf = ebv[CONFORMATION].mean()
deviation_from_national = conf_by_province - national_avg_conf

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for ax, trait in zip(axes, CONFORMATION):
    data = deviation_from_national[trait].sort_values()
    colors = ["#B23A3A" if v < 0 else "#2E5B4D" for v in data]
    ax.barh(data.index, data.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(trait)
    ax.set_xlabel("vs. national average")

plt.suptitle("Conformation strengths (green) and weaknesses (red) by province - EBV animals", y=1.03)
plt.tight_layout()
plt.show()


**Findings:** Quebec, Alberta, and Ontario are consistently above the
national average across all five conformation traits. Prince Edward Island is
below average on every trait - the weakest overall conformation profile.
Newfoundland and Labrador is especially weak in F&L (feet and legs).


## Step 11 - Correlation between production/components and conformation

Instead of a heatmap, this is a single sorted bar chart of every
component-vs-trait pair. Dashed gray lines mark ±0.5, the usual threshold for
"worth reporting."


In [ ]:
COMPONENT_KG = ["Milk", "Fat", "Prot"]
COMPONENT_PCT = ["%F", "%P"]

corr = ebv[COMPONENT_KG + COMPONENT_PCT + CONFORMATION].corr()

pairs = []
for comp in COMPONENT_KG + COMPONENT_PCT:
    for trait in CONFORMATION:
        pairs.append({"Pair": f"{comp} vs. {trait}", "Correlation": round(corr.loc[comp, trait], 2)})
pairs_df = pd.DataFrame(pairs).sort_values("Correlation", key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#B23A3A" if v < 0 else "#2E5B4D" for v in pairs_df["Correlation"]]
ax.barh(pairs_df["Pair"], pairs_df["Correlation"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(-0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Correlation coefficient (r)")
ax.set_title("Correlation: production/components vs. conformation traits")
plt.tight_layout()
plt.show()


In [ ]:
# Stability check: does the strongest pair hold up across random halves of the data?
def check_stability(df, col_a, col_b, n_splits=5):
    results = []
    for i in range(n_splits):
        sample = df.sample(frac=0.5, random_state=i)
        r = sample[[col_a, col_b]].corr().iloc[0, 1]
        results.append(round(r, 3))
    return results

strongest = pairs_df.iloc[-1]
comp, trait = strongest["Pair"].split(" vs. ")
print(f"Strongest pair found: {comp} vs. {trait} (r = {strongest['Correlation']})")
print(f"Across 5 random halves: {check_stability(ebv, comp, trait)}")

r_stat, p_value = stats.pearsonr(ebv[comp].dropna(), ebv[trait].dropna())
print(f"Pearson r = {r_stat:.3f}, p-value = {p_value:.4f}")


**Findings:** none of the correlations cross the ±0.5 "worth reporting"
threshold. The strongest was Milk vs. Dairy Strength (DS) at only r ≈ 0.26 -
weak. **Conclusion: production traits and conformation traits are
essentially independent in this dataset** - which matches how these indices
are deliberately designed in genetic evaluation systems, so that selecting
for production doesn't sacrifice structure, and vice versa. With ~1,900
animals, even tiny correlations can show a "significant" p-value - that's
why the magnitude (r) matters more than the p-value here.


## Step 12 - Conformation priorities/trends by province

Which conformation trait does each province stand out in the most, relative
to the other provinces?


In [ ]:
z_scores = (conf_by_province - conf_by_province.mean()) / conf_by_province.std()
standout_trait = z_scores.idxmax(axis=1)
standout_strength = z_scores.max(axis=1).round(2)

priority_table = pd.DataFrame({
    "Standout Trait": standout_trait,
    "Z-score": standout_strength,
}).sort_values("Z-score", ascending=False)
priority_table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2E5B4D" if z > 0 else "#B23A3A" for z in priority_table["Z-score"]]
ax.barh(priority_table.index, priority_table["Z-score"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
for i, (trait, z) in enumerate(zip(priority_table["Standout Trait"], priority_table["Z-score"])):
    ax.text(z, i, f" {trait}", va="center", fontsize=9)
ax.set_xlabel("How far above/below the province's own average (z-score)")
ax.set_title("Each province's standout conformation trait")
plt.tight_layout()
plt.show()


**Findings:** Alberta, Ontario, and Saskatchewan stand out most in MS
(Mammary System). Quebec stands out most in F&L (Feet & Legs) - the most
trustworthy signal here given its larger sample size. No single trait
dominates nationally; treat this as a directional signal, not a confirmed
regional breeding strategy, given the small sample sizes per province.


## Step 12.5 - Conformation vs. production priority, by province

A different composite from Step 12: this contrasts conformation traits
(Conf, MS, F&L, DS, RP) against health/fertility traits (HWI, RI, MI, EI),
both standardized so they're directly comparable, to see which side of
that trade-off each province's top-400 leans toward. This uses each
province's full 400-animal group (not filtered by `LPI Code`), so all 10
provinces are equally usable here.


In [ ]:
HEALTH_FERTILITY = ["HWI", "RI", "MI", "EI"]

conf_by_province_full = clean.groupby("Province Name", observed=True)[CONFORMATION].mean()
conf_z_full = (conf_by_province_full - conf_by_province_full.mean()) / conf_by_province_full.std()

health_by_province = clean.groupby("Province Name", observed=True)[HEALTH_FERTILITY].mean()
health_z = (health_by_province - health_by_province.mean()) / health_by_province.std()

priority_score = (conf_z_full.mean(axis=1) - health_z.mean(axis=1)).round(2).sort_values(ascending=False)
priority_score.name = "Conformation (+) vs. Health/Fertility (-)"
priority_score.to_frame()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#2E5B4D" if v > 0 else "#B23A3A" for v in priority_score]
ax.barh(priority_score.index, priority_score.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Leans toward Conformation (+) vs. Health/Fertility (-)")
ax.set_title("Conformation vs. health/fertility priority, all 10 provinces")
plt.tight_layout()
plt.show()


**Reading this:** every province is included, ranked on one
comparable scale. Ontario leans most toward conformation; Manitoba leans
most toward health/fertility, more than three times as strongly as any
other province. This should be read as **"which traits accompany a high
LPI ranking in each province's top 400,"** not necessarily a deliberate,
stated breeding policy - the data shows the pattern but not the intent
behind it.


## Step 13 - Does having an EBV actually matter? (PA vs. EBV, 2+ years)

All 10 provinces now show a plausible `LPI Code` split (see the data notes in Step 4), so no provinces need to be excluded from this comparison.


In [ ]:
VALID_PROVINCE_CODES = ["AB", "BC", "MB", "NB", "NL", "NS", "ON", "PEI", "QC", "SK"]

available_codes = [c for c in VALID_PROVINCE_CODES if c in mature["Province"].unique()]
missing = set(VALID_PROVINCE_CODES) - set(available_codes)
if missing:
    print(f"Warning: these province codes are not in your data: {missing}")

valid = mature[mature["Province"].isin(available_codes)]
print(f"Rows in valid subset: {len(valid)}")
print(f"LPI Code groups found: {sorted(valid['LPI Code'].unique())}")


In [ ]:
KEY_TRAITS = ["LPI", "Pro$", "Milk", "Fat", "Prot", "Conf"]
comparison = valid.groupby("LPI Code")[KEY_TRAITS].mean().round(1)

if "EBV" in comparison.index and "PA" in comparison.index:
    comparison.loc["Difference (EBV - PA)"] = (comparison.loc["EBV"] - comparison.loc["PA"]).round(1)
    print(f"Sample sizes -> EBV: {(valid['LPI Code']=='EBV').sum()}, PA: {(valid['LPI Code']=='PA').sum()}")
else:
    print("WARNING: one of the groups (EBV or PA) is missing from 'valid' - check Step 1 for missing files.")

comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
comparison.loc[["EBV", "PA"]].T.plot(kind="bar", ax=ax, color=["#2E5B4D", "#C9A66B"])
ax.set_title("EBV vs. PA animals (2+ years, reliable provinces only)")
ax.set_ylabel("Average value")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


**Findings:** animals with an EBV score higher across every
trait checked - LPI, Pro$, Milk, Fat, Protein, and Conf. This is consistent
with the idea that an EBV (official phenotype-based evaluation: production
+ classification records) carries information beyond pedigree alone. It's
a correlational comparison though, not a controlled experiment - it may
partly reflect that farmers collect full records on animals they already
expect to perform well.


## Step 14 - Correlation assumptions: are we allowed to trust Pearson's r?

Pearson's r assumes roughly normal data, a linear relationship, and no
outliers dominating the result. Checked here instead of assumed.


In [ ]:
for col in ["LPI", "Milk", "Fat", "Prot", "Conf"]:
    stat, p = stats.normaltest(clean[col].dropna())
    skew = stats.skew(clean[col].dropna())
    kurt = stats.kurtosis(clean[col].dropna())
    print(f"{col}: normaltest p={p:.4g}, skew={skew:.2f}, kurtosis={kurt:.2f}")

print()
for col in ["LPI", "Milk", "Fat", "Prot"]:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((clean[col] < lower) | (clean[col] > upper)).sum()
    print(f"{col}: {n_out} outliers ({n_out/len(clean)*100:.1f}%)")


**Findings:** formal normality tests reject normality (p<0.001) for
every trait checked, but this is expected with ~4,000 observations, where
even trivial deviations become "significant." Skew (-0.27 to 0.04) and
kurtosis (-0.80 to 0.22) are both small, and outlier share is low (0-1%) -
Pearson's r is reliable enough to use here.


## Step 15 - Regression diagnostics: VIF, residuals, homoscedasticity

This step both runs a standardized regression of LPI on its component
traits and validates it. Important context before reading the results:
per Lactanet's own published LPI structure ("Which Index is Right for My
Herd?", Lactanet, March 2025), LPI is a weighted sum of six subindexes
(Production Index 40%, Longevity and Type Index 32%, Health and Welfare
Index 8%, Reproduction Index 10%, Milkability Index 5%, Environmental
Impact Index 5%). Nearly every trait used below is a direct structural
ingredient of one of those subindexes, not an independent variable being
tested against LPI. The regression mainly demonstrates that known,
published structure rather than revealing something new.


In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

TRAITS = ["Milk", "Fat", "Prot", "Conf", "MS", "F&L", "DS", "RP", "HWI", "RI", "MI", "EI", "SCS"]
reg_df = clean.dropna(subset=TRAITS + ["LPI"]).copy()
X_std = (reg_df[TRAITS] - reg_df[TRAITS].mean()) / reg_df[TRAITS].std()
y_std = (reg_df["LPI"] - reg_df["LPI"].mean()) / reg_df["LPI"].std()
X_const = sm.add_constant(X_std)

model = sm.OLS(y_std, X_const).fit()
print(f"R-squared: {model.rsquared:.3f}")

vif_data = pd.DataFrame({
    "Trait": X_std.columns,
    "VIF": [variance_inflation_factor(X_std.values, i) for i in range(X_std.shape[1])],
}).sort_values("VIF", ascending=False)
print(vif_data)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.scatter(model.fittedvalues, model.resid, alpha=0.3, s=10, color="#2E5B4D")
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("Fitted values")
ax1.set_ylabel("Residuals")
ax1.set_title("Residual plot")

sm.qqplot(model.resid, line="45", fit=True, ax=ax2)
ax2.set_title("QQ plot of residuals")

plt.tight_layout()
plt.show()

bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
stat, p_norm = stats.normaltest(model.resid)
print(f"Breusch-Pagan (homoscedasticity): p={bp_p:.4g}")
print(f"Residual normality test: p={p_norm:.4g}, skew={stats.skew(model.resid):.2f}")


**Findings:** R²=0.984, which given the subindex context above should
be read as confirming LPI's published formula structure, not as an
independent discovery. **VIF for `Conf` and `MS` is severe (>40)** -
expected, since `Conf` is itself built from `MS`/`F&L`/`DS`/`RP` by
Lactanet's own scoring, making it mathematically redundant with them, not
an independent predictor. Breusch-Pagan confirms heteroscedasticity
(p<0.001) - robust standard errors (HC3) should be used for any p-values
quoted from this model. Residual normality is formally rejected but the
skew/kurtosis are small, not a practical concern. One coefficient needs a
specific correction: Milk's standardized coefficient here is close to
zero, but Lactanet's own published correlation between Milk Yield and LPI
is a real, moderate 0.43. The near-zero coefficient reflects
multicollinearity between Milk, Fat, and Protein (all three come from the
same PI subindex), not evidence that milk volume is unimportant to LPI.


## Step 16 - ANOVA + Tukey HSD across all 10 provinces

A t-test compares 2 groups; comparing all 10 provinces at once calls for
ANOVA (does *any* province differ) followed by Tukey HSD (*which* pairs
differ, without inflating false positives from 45 separate tests).


In [ ]:
anova_results = []
for trait in ["LPI", "Milk", "Fat", "Prot", "Conf"]:
    groups = [g[trait].dropna().values for _, g in clean.groupby("Province Name", observed=True)]
    f_stat, p_value = stats.f_oneway(*groups)
    anova_results.append({"Trait": trait, "F-statistic": round(f_stat, 2), "p-value": p_value})

pd.DataFrame(anova_results)


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(clean["LPI"], clean["Province Name"], alpha=0.05)
tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
print(f"Pairs NOT significantly different (out of {len(tukey_df)} total):")
tukey_df[tukey_df["reject"] == False]


**Findings:** ANOVA is significant (p<0.001) for every trait checked -
province is a real source of variation. Of 45 province pairs, only
**Prince Edward Island vs. Saskatchewan** is not significantly different
(p=1.0, mean difference only 2.6 points) - every other pair of provinces
differs significantly on LPI.


## Step 17 - Confidence intervals (95% CI) for average LPI by province


In [ ]:
def confidence_interval_95(series):
    n = len(series)
    mean = series.mean()
    sem = series.std() / (n ** 0.5)
    margin = sem * stats.t.ppf(0.975, n - 1)
    return mean, mean - margin, mean + margin

ci_results = []
for province, group in clean.groupby("Province Name", observed=True):
    mean, lower, upper = confidence_interval_95(group["LPI"])
    ci_results.append({"Province": province, "Mean LPI": round(mean, 1),
                        "95% CI Lower": round(lower, 1), "95% CI Upper": round(upper, 1)})

ci_df = pd.DataFrame(ci_results).sort_values("Mean LPI", ascending=False)
ci_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(ci_df))
ax.errorbar(ci_df["Mean LPI"], y_pos,
            xerr=[ci_df["Mean LPI"] - ci_df["95% CI Lower"], ci_df["95% CI Upper"] - ci_df["Mean LPI"]],
            fmt="o", color="#2E5B4D", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(ci_df["Province"])
ax.set_xlabel("Mean LPI (95% CI)")
ax.set_title("Average LPI by province with 95% confidence intervals")
plt.tight_layout()
plt.show()


## Step 18 - PCA: how do provinces group by overall genetic profile?


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

RANDOM_SEED = 42 # fixed for reproducibility across every random process below

TRAITS_ALL = ["LPI", "Pro$", "Milk", "Fat", "Prot", "%F", "%P", "Conf", "MS", "F&L", "DS", "RP", "HWI", "RI", "MI", "EI", "SCS"]
province_profiles = clean.groupby("Province Name", observed=True)[TRAITS_ALL].mean()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(province_profiles)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pcs = pca.fit_transform(X_scaled)
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}% of variance, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=province_profiles.index)

km = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=10)
pca_df["Cluster"] = km.fit_predict(X_scaled)
pca_df.sort_values("Cluster")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors_map = {0: "#2E5B4D", 1: "#C9A66B", 2: "#B23A3A"}
for cluster_id in sorted(pca_df["Cluster"].unique()):
    subset = pca_df[pca_df["Cluster"] == cluster_id]
    ax.scatter(subset["PC1"], subset["PC2"], s=200, color=colors_map[cluster_id], label=f"Cluster {cluster_id}")
    for name, row in subset.iterrows():
        ax.annotate(name, (row["PC1"], row["PC2"]), fontsize=9, xytext=(5, 5), textcoords="offset points")

ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%) - overall strength (most traits above/below average together)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%) - feed efficiency vs. conformation trade-off")
ax.set_title("PCA: how do provinces group by genetic profile?")
ax.legend()
plt.tight_layout()
plt.show()


## Step 19 - Hierarchical clustering (cross-check for Step 18)


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

Z = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(11, 6))
dendrogram(Z, labels=province_profiles.index.tolist(), ax=ax, color_threshold=6)
ax.set_ylabel("Distance (Ward's method)")
ax.set_title("Hierarchical clustering of provinces by genetic profile")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Findings (Steps 18-19) - what the numbers actually show:**

Two axes capture most of the real structure here (87% of it combined), so
this is a genuinely good summary, not an oversimplification.

**Axis 1 (60% of the pattern): overall strength.** Nearly every trait we
measured - LPI, profitability, milk/fat/protein, conformation, health index
 - moves up or down *together* on this axis. A province high on this axis
is simply scoring above the dataset average across almost the whole board;
a province low on it is below average across almost the whole board. Dairy
strength barely moves on this axis at all, and feed efficiency only weakly.

**Axis 2 (27% of the pattern): a real trade-off between feed efficiency and
conformation.** This axis is dominated by feed efficiency pulling one way,
and dairy strength, rump, overall conformation, and mammary system pulling
the other way - confirmed directly in the raw averages: Manitoba has the
highest feed-efficiency score of any province (525.6) and is the *only*
province with a negative average Dairy Strength (-1.4), while Ontario and
Quebec have the lowest feed-efficiency scores paired with the highest
conformation scores. This is a real pattern in the data, not a labeling
choice.

**What each province's position means (top-400-by-LPI group, not the
province's full cattle population):**

- **Quebec, Ontario:** high on "overall strength" - above dataset average
  on most traits at once - and lean toward the conformation side of the
  efficiency/conformation trade-off, more strongly than any other province.
- **Alberta:** generally above/near dataset average on "overall strength,"
  close to neutral on the efficiency/conformation trade-off.
- **British Columbia, Prince Edward Island:** near or above dataset
  average on "overall strength," leaning toward the feed-efficiency side.
- **Manitoba:** above dataset average on "overall strength" (comparable to
  Alberta/BC), but by far the most extreme on the efficiency/conformation
  axis - highest feed efficiency and lowest conformation-related scores of
  any province.
- **New Brunswick, Nova Scotia, Newfoundland & Labrador, Saskatchewan:**
  all below dataset average on "overall strength" - the main thing they
  share. They cluster together primarily because of that shared
  below-average first axis, not because they're identical on the second.

**Why the groups land where they do:** both clustering methods (KMeans and
hierarchical) group provinces mainly by combining position on both axes.
Quebec and Ontario separate from Manitoba/BC/PEI/Alberta specifically
because of how strongly they lean toward conformation on the second axis,
even though several provinces in both groups score similarly on "overall
strength." Manitoba remains the most extreme province on the
efficiency/conformation axis of any in the dataset.

**What this does and doesn't mean:**

- **Clustering groups provinces by how similar their overall genetic
  *profile shape* is - it is not a ranking, and it does not mean one
  cluster is genetically "better" or "worse" than another.** The
  higher-strength provinces score above dataset average on most individual
  traits, which is a factual difference - but "different profile" and
  "superior/inferior" are not the same claim, and this analysis does not
  support the second one.
- **Every statement above describes each province's top-400-by-LPI group,
  not its overall dairy cattle population.** A province clustering as
  "lower overall strength" here says nothing about the average commercial
  herd in that province - only about how its elite, LPI-ranked animals
  compare to other provinces' elite groups.


## Step 20 - Within-animal trait balance (a different kind of "homogeneity")

Different question from Step 10's province-level homogeneity: here we ask
whether *individual animals* tend to be strong in some conformation traits
and weak in others, and whether that varies by province.


In [ ]:
z_conformation = (clean[CONFORMATION] - clean[CONFORMATION].mean()) / clean[CONFORMATION].std()
clean["Trait Imbalance (within animal)"] = z_conformation.std(axis=1)

imbalance_by_province = (
    clean.groupby("Province Name", observed=True)["Trait Imbalance (within animal)"]
    .mean().round(3).sort_values()
)
imbalance_by_province


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
imbalance_by_province.plot(kind="bar", ax=ax, color="#2E5B4D")
ax.set_ylabel("Avg. within-animal trait imbalance (lower = more balanced)")
ax.set_title("How evenly matched are an animal's own conformation traits, by province?")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Findings:** this is a genuinely different question from Step 10.
Quebec led on *between-animal* homogeneity (consistent LPI scores across
its 400 animals), but is only mid-pack here - **British Columbia and New
Brunswick have the most internally balanced individual animals**
(no single trait wildly outperforming the others within the same cow), while
**Newfoundland & Labrador and PEI have the least balanced animals** - the
same two provinces that showed up weakest in Step 10's conformation
strengths/weaknesses chart.


## Step 20.5 - Does information integration relate to elite population strength?

A more useful question than "which province wins" is: **do provinces with
more genomically tested animals in their elite population show stronger or
more homogeneous elite populations?** This uses the `GS` field, which
(unlike `LPI Code`) looks trustworthy across all 10 provinces - checked
directly below, not assumed.


In [ ]:
integration_rows = []
for province, group in clean.groupby("Province Name", observed=True):
    pct_any_genomic = group["Is Genomic"].mean() * 100
    integration_rows.append({
        "Province": province,
        "% Genomic Representation": round(pct_any_genomic, 1),
        "Mean LPI": round(group["LPI"].mean(), 1),
        "LPI CV%": round(group["LPI"].std() / group["LPI"].mean() * 100, 2),
    })

integration_df = pd.DataFrame(integration_rows).sort_values("% Genomic Representation", ascending=False)
integration_df


In [ ]:
r_lpi, p_lpi = stats.pearsonr(integration_df["% Genomic Representation"], integration_df["Mean LPI"])
rho_lpi, prho_lpi = stats.spearmanr(integration_df["% Genomic Representation"], integration_df["Mean LPI"])
print(f"All 10 provinces -> Pearson r={r_lpi:.3f} (p={p_lpi:.4f}), Spearman rho={rho_lpi:.3f} (p={prho_lpi:.4f})")

r_cv, p_cv = stats.pearsonr(integration_df["% Genomic Representation"], integration_df["LPI CV%"])
print(f"vs. LPI CV% (consistency): r={r_cv:.3f} (p={p_cv:.4f})")

without_qc = integration_df[integration_df["Province"] != "Quebec"]
r_robust, p_robust = stats.pearsonr(without_qc["% Genomic Representation"], without_qc["Mean LPI"])
print(f"Robustness check, excluding Quebec (most extreme point): r={r_robust:.3f} (p={p_robust:.4f}, n={len(without_qc)})")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(integration_df["% Genomic Representation"], integration_df["Mean LPI"], s=120, color="#2E5B4D")
for _, row in integration_df.iterrows():
    ax.annotate(row["Province"], (row["% Genomic Representation"], row["Mean LPI"]), fontsize=9, xytext=(6, 6), textcoords="offset points")

z = np.polyfit(integration_df["% Genomic Representation"], integration_df["Mean LPI"], 1)
x_line = np.linspace(integration_df["% Genomic Representation"].min(), integration_df["% Genomic Representation"].max(), 50)
ax.plot(x_line, np.poly1d(z)(x_line), color="#888780", linestyle="--", linewidth=1.5)

ax.set_xlabel("% of top-400 animals with a genomic test on file")
ax.set_ylabel("Mean LPI")
ax.set_title(f"Genomic representation vs. elite population strength (r={r_lpi:.2f}, p={p_lpi:.3f}, n=10)")
plt.tight_layout()
plt.show()


**Findings:** across all 10 provinces, genomic representation is
associated with average LPI (r=0.73, p=0.016; Spearman ρ=0.81, p=0.005,
confirming the relationship isn't driven by outliers in the raw values).
It holds up removing Quebec, the most extreme point (r=0.71, p=0.033,
n=9). **Genomic representation is not associated with LPI consistency**
(r=0.01, not significant) - that specific claim, found in earlier drafts
of this analysis, does not hold with the corrected data and has been
dropped.

**What this does and doesn't support:** provinces differ not only in
genetic merit, but in the *composition of information* behind their elite
populations' evaluations. Because genomic testing, milk recording, and
classification provide complementary rather than redundant information,
these differences may represent opportunities to strengthen how
consistently all three sources are combined across regions. **This
dataset cannot determine why these differences exist** - it contains no
information about producer education, cost, service access, or awareness,
so no claim about producers "not understanding the value" of any service
is supported here. What the data does support: these regional differences
in information composition are real, measurable, and associated with
observable differences in the elite population's strength - which may be
useful for province-specific extension or communication about combining
genomic testing with production and classification records, without
asserting any particular reason why adoption differs today.


## Step 21 - Reproducibility notes

- `RANDOM_SEED = 42` is used everywhere randomness appears (KMeans, PCA) so
  re-running this notebook produces identical results.
- Library versions used: run `pip freeze > requirements.txt` in this
  project's environment to capture the exact versions for full
  reproducibility elsewhere.
- The data loading and cleaning steps involve no randomness - the same
  input files always produce the same cleaned dataset.


## Final summary

- Loaded and consolidated 10 provincial files (4,000 animals total).
- Cleaned data types and made `Act.`/`GS` codes explicit.
- Calculated national and provincial averages, and genomic testing rates.
- Compared top animals by component (kg and %) and by conformation, by
  province, using only animals with their own EBV data for a fair comparison.
- Mapped conformation strengths/weaknesses by province with readable bar
  charts (no heatmaps).
- Checked correlation between production and conformation, with assumptions
  (normality, linearity, outliers) verified first: essentially no
  relationship found (all |r| < 0.3).
- Validated the LPI regression with VIF, residual/QQ plots, and a
  homoscedasticity test - found and addressed multicollinearity in `Conf`.
- Ran ANOVA + Tukey HSD across all 10 provinces, and 95% confidence
  intervals for each province's average LPI.
- Used PCA and two independent clustering methods to find natural province
  groupings.
- Measured within-animal trait balance, a different question from
  between-animal homogeneity.
- Compared PA-only vs. EBV animals in the provinces with trustworthy data:
  EBV animals score higher across the board, and genomic testing shows a
  much larger effect on estimate *reliability* than on the estimate itself.
- Tested whether information integration (genomic representation) associates
  with elite-population strength across provinces: yes, a moderate,
  robustness-checked correlation (r=0.73, p=0.016, n=10 - no provinces
  excluded).

### Known data limitations (keep these in mind for your conclusions)
- Each province file is a **top-400-by-LPI list**, not the full population.
- All averages in this notebook are **dataset averages** (from these 4,000
  records only), not Lactanet's official national average.
- This is a **snapshot in time**: Lactanet's evaluations are released
  periodically, and top-400 membership shifts as new births, phenotype,
  and genomic data arrive. Every animal in this dataset is confirmed
  unique (0 overlap across all 10 provinces), and all 10 provinces show a
  plausible `LPI Code` split.
- Age is approximate (birth year only, no exact date), and the "2+ years"
  filter assumes but does not confirm an animal has its own EBV data.
- `GS` (genomic tested) can't distinguish "never tested" from "result
  still pending," especially for the youngest animals.
